# 📊 Attrition Risk Dashboard
Loads the pretrained model from `01_Model_Builder.ipynb` and provides:
1. **Validation Results** — AUC, Recall, Precision on historical data
2. **2026 Live Risk Scores** — Current employee risk profile
3. **Action Lists** — Saveable Stars, High Risk Employees
4. **Breakdowns** — By Management Level, Org, Region
5. **CSV Export**

**Prerequisites:** Run `01_Model_Builder.ipynb` first.

In [ ]:
!pip install pandas numpy xgboost scikit-learn matplotlib seaborn pyarrow joblib

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display, HTML
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 50)
plt.style.use('ggplot')

# ===================== PATHS (must match Model Builder) =====================
BASE_DIR = os.getcwd()
PATH_OUTPUT = os.path.join(BASE_DIR, '..', '04_Master_Data')
PATH_MODEL_OUTPUT = os.path.join(BASE_DIR, 'model_artifacts')

TARGET_COLUMN = 'Will_Leave_Within_Horizon'
PREDICTION_HORIZON_MONTHS = 6

BASE_FEATURES = [
    'Total_Tenure_Years', 'Time_In_Title_Current', 'Cycles_In_Level',
    'Is_Stuck_In_Level', 'Days_Since_Last_Promotion', 'Consecutive_Low_Perf_Years',
    'Performance_Score', 'Is_High_Performer', 'Performance_Drop_Flag',
    'Perf_Score_Rolling_Avg_3Yr', 'Perf_Trajectory_Slope',
    'Peer_Promo_Rate', 'Peer_Pressure_Flag',
    'Manager_Past_Churn_Count', 'Manager_Performance_Score',
    'Performer_vs_Mgr_Score', 'Manager_Changed_Recently', 'Manager_Span_Of_Control',
    'Job_Family_Churn_Rate', 'Org_Level_Churn_Rate',
    'Is_Lateral_Hire', 'Is_Campus_Hire', 'Has_International_Assignment',
    'Is_Acquisition_Employee', 'FTE_Value', 'Is_Reorg_Affected', 'Is_Transfer_Recent',
    'Month_Sin', 'Month_Cos', 'Is_Post_Bonus_Window',
]
CATEGORICAL_FEATURES = ['Management_Level_Agg']

print('✅ Dashboard config loaded.')

---
## 1. Load Pretrained Model & Data

In [ ]:
# Load pretrained model
model_path = os.path.join(PATH_MODEL_OUTPUT, 'xgboost_model.joblib')
if not os.path.exists(model_path):
    raise FileNotFoundError(f'No pretrained model found at:\n  {model_path}\nRun 01_Model_Builder.ipynb first.')

xgb = joblib.load(model_path)
print(f'✅ Pretrained model loaded from: {model_path}')

# Load feature names (saved by model builder)
feat_names_path = os.path.join(PATH_MODEL_OUTPUT, 'feature_names.json')
if os.path.exists(feat_names_path):
    with open(feat_names_path, 'r') as f:
        trained_feature_names = json.load(f)
    print(f'✅ Feature names loaded: {len(trained_feature_names)} features')
else:
    trained_feature_names = None
    print('   ⚠️ feature_names.json not found. Will infer from model.')

# Load featured data
data_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.parquet')
if not os.path.exists(data_path):
    data_path = os.path.join(PATH_OUTPUT, 'Model_Ready_Data.csv')

if os.path.exists(data_path):
    df = pd.read_parquet(data_path) if data_path.endswith('.parquet') else pd.read_csv(data_path, low_memory=False)
    df['Snapshot_Date'] = pd.to_datetime(df['Snapshot_Date'])
    print(f'✅ Data loaded: {len(df):,} rows | Years: {sorted(df["Year"].unique())}')
else:
    raise FileNotFoundError(f'No data found. Run 01_Model_Builder.ipynb first.\nLooked at: {data_path}')

In [ ]:
def prepare_data(df, prediction_year, trained_features=None):
    """Prepare X_test and test_df for a given prediction year using the pretrained model's features."""
    all_feats = BASE_FEATURES + CATEGORICAL_FEATURES
    avail = [f for f in all_feats if f in df.columns]
    X_all = pd.get_dummies(df[avail], columns=CATEGORICAL_FEATURES, drop_first=True)
    y_all = df[TARGET_COLUMN]
    
    # Test set: latest snapshot per employee in prediction_year
    test_raw = df[df['Year'] == prediction_year].sort_values(
        ['Employee_ID', 'Snapshot_Date'], ascending=[True, False]
    )
    test_unique = test_raw.drop_duplicates(subset=['Employee_ID'], keep='first')
    X_test = pd.get_dummies(test_unique[avail], columns=CATEGORICAL_FEATURES, drop_first=True)
    y_test = test_unique[TARGET_COLUMN]
    
    # Align to trained model features
    if trained_features is not None:
        for c in trained_features:
            if c not in X_test.columns:
                X_test[c] = 0
        X_test = X_test[trained_features].fillna(0)
    else:
        X_test = X_test.fillna(0)
    
    return X_test, y_test, test_unique

print('✅ Data preparation function ready.')

---
## 2. Validation Year Results ("The Exam")

In [ ]:
# Auto-select validation year (latest year with positive targets)
years_pos = df.groupby('Year')[TARGET_COLUMN].sum()
years_with = years_pos[years_pos > 0]
VAL_YEAR = int(years_with.index.max()) if not years_with.empty else df['Year'].max()

X_test_val, y_test_val, test_df_val = prepare_data(df, VAL_YEAR, trained_feature_names)

threshold = 0.5
y_prob_val = xgb.predict_proba(X_test_val)[:, 1]
y_pred_val = (y_prob_val > threshold).astype(int)

auc = roc_auc_score(y_test_val, y_prob_val) if y_test_val.sum() > 0 else 0
rec = recall_score(y_test_val, y_pred_val, zero_division=0)
prec = precision_score(y_test_val, y_pred_val, zero_division=0)

print('=' * 55)
print(f'   📋 VALIDATION RESULTS ({VAL_YEAR})')
print('=' * 55)
print(f'   AUC:       {auc:.4f}')
print(f'   Recall:    {rec:.4f}  (caught {rec:.0%} of actual leavers)')
print(f'   Precision: {prec:.4f}  ({prec:.0%} of flags were correct)')
print(f'   Population: {len(X_test_val):,} | Actual leavers: {int(y_test_val.sum())} | Flagged: {int(y_pred_val.sum())}')

cm = confusion_matrix(y_test_val, y_pred_val)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix ({VAL_YEAR})')
axes[1].hist(y_prob_val[y_test_val==0], bins=50, alpha=0.6, label='Stayers', color='teal')
axes[1].hist(y_prob_val[y_test_val==1], bins=50, alpha=0.6, label='Leavers', color='salmon')
axes[1].axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
axes[1].set_title('Risk Scores: Stayers vs Leavers'); axes[1].legend()
plt.tight_layout()
plt.show()

---
## 3. Feature Importance: Top Drivers

In [ ]:
feat_display = trained_feature_names if trained_feature_names else [f'f{i}' for i in range(len(xgb.feature_importances_))]
imps = pd.DataFrame({'Feature': feat_display, 'Importance': xgb.feature_importances_}).sort_values('Importance', ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=imps, palette='viridis')
plt.title('Top 15 Drivers of Attrition (Pretrained Model)')
plt.tight_layout()
plt.show()

---
## 4. 🚀 2026 Live Risk Profile

In [ ]:
LIVE_YEAR = 2026
X_test_live, y_test_live, test_df_live = prepare_data(df, LIVE_YEAR, trained_feature_names)

y_prob_live = xgb.predict_proba(X_test_live)[:, 1]
test_df_live = test_df_live.copy()
test_df_live['Risk_Score'] = y_prob_live
test_df_live['Risk_Category'] = pd.cut(y_prob_live, bins=[0, 0.3, 0.6, 1.0], labels=['🟢 Low', '🟡 Medium', '🔴 High'])
test_df_live['Predicted_Leaver'] = (y_prob_live > threshold).astype(int)

print(f'✅ Risk scores generated for {len(test_df_live):,} active employees ({LIVE_YEAR})')
print(f'\n   Risk Breakdown:')
display(test_df_live['Risk_Category'].value_counts().to_frame('Count'))

plt.figure(figsize=(10, 4))
sns.histplot(test_df_live['Risk_Score'], bins=50, kde=True, color='teal')
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold {threshold}')
plt.title(f'{LIVE_YEAR} Employee Risk Distribution')
plt.xlabel('Attrition Risk (6-month window)')
plt.legend()
plt.show()

---
## 5. 📋 Action Lists

In [ ]:
show_cols = ['Employee_ID', 'Risk_Score', 'Risk_Category', 'Management_Level_Agg',
             'Total_Tenure_Years', 'Performance_Score', 'Level 2 Org', 'Level 3 Org',
             'Region', 'Country']
show_cols = [c for c in show_cols if c in test_df_live.columns]

# Saveable Stars
print('⭐ SAVEABLE STARS — High Performers (Rating 4-5) at High Risk')
print('These are your top talent the model thinks may leave.\n')
stars = test_df_live[
    (test_df_live['Risk_Score'] > 0.5) &
    (test_df_live.get('Performance_Score', pd.Series(dtype=float)).isin([4, 5]))
].sort_values('Risk_Score', ascending=False)
print(f'Total Saveable Stars: {len(stars)}')
if not stars.empty:
    display(stars[show_cols].head(30))
else:
    print('None found at current threshold.')

In [ ]:
# High Risk List
print('🔴 HIGH RISK EMPLOYEES (Risk > 0.7)')
high_risk = test_df_live[test_df_live['Risk_Score'] > 0.7].sort_values('Risk_Score', ascending=False)
print(f'Total High Risk: {len(high_risk)}')
if not high_risk.empty:
    display(high_risk[show_cols].head(30))
else:
    print('No employees above 0.7 risk.')

---
## 6. 📊 Risk Breakdowns

In [ ]:
# By Management Level
if 'Management_Level_Agg' in test_df_live.columns:
    print('📊 RISK BY MANAGEMENT LEVEL')
    lvl = test_df_live.groupby('Management_Level_Agg').agg(
        Headcount=('Employee_ID','count'), Avg_Risk=('Risk_Score','mean'),
        High_Risk=('Predicted_Leaver','sum')
    ).reset_index()
    lvl['High_Risk_Pct'] = (lvl['High_Risk'] / lvl['Headcount'] * 100).round(1)
    display(lvl.sort_values('Avg_Risk', ascending=False))

    plt.figure(figsize=(10, 4))
    sns.barplot(x='Avg_Risk', y='Management_Level_Agg', data=lvl.sort_values('Avg_Risk', ascending=False), palette='Reds_r')
    plt.title('Average Risk by Management Level')
    plt.xlabel('Average Risk Score')
    plt.tight_layout()
    plt.show()

In [ ]:
# By Level 2 Org
if 'Level 2 Org' in test_df_live.columns:
    print('📊 RISK BY LEVEL 2 ORG')
    org = test_df_live.groupby('Level 2 Org').agg(
        Headcount=('Employee_ID','count'), Avg_Risk=('Risk_Score','mean'),
        High_Risk=('Predicted_Leaver','sum')
    ).reset_index()
    org['High_Risk_Pct'] = (org['High_Risk'] / org['Headcount'] * 100).round(1)
    display(org.sort_values('Avg_Risk', ascending=False))

In [ ]:
# By Region
if 'Region' in test_df_live.columns:
    print('📊 RISK BY REGION')
    reg = test_df_live.groupby('Region').agg(
        Headcount=('Employee_ID','count'), Avg_Risk=('Risk_Score','mean'),
        High_Risk=('Predicted_Leaver','sum')
    ).reset_index()
    reg['High_Risk_Pct'] = (reg['High_Risk'] / reg['Headcount'] * 100).round(1)
    display(reg.sort_values('Avg_Risk', ascending=False))

---
## 7. 💾 Export Risk Scores

In [ ]:
export_path = os.path.join(PATH_MODEL_OUTPUT, f'Risk_Scores_{LIVE_YEAR}.csv')
export_cols = show_cols + ['Predicted_Leaver']
export_cols = [c for c in export_cols if c in test_df_live.columns]
test_df_live[export_cols].to_csv(export_path, index=False)
print(f'✅ Exported {len(test_df_live):,} employee risk scores to:')
print(f'   {export_path}')

if not stars.empty:
    stars_path = os.path.join(PATH_MODEL_OUTPUT, f'Saveable_Stars_{LIVE_YEAR}.csv')
    stars[export_cols].to_csv(stars_path, index=False)
    print(f'\n✅ Saveable Stars exported to: {stars_path}')

if not high_risk.empty:
    hr_path = os.path.join(PATH_MODEL_OUTPUT, f'High_Risk_{LIVE_YEAR}.csv')
    high_risk[export_cols].to_csv(hr_path, index=False)
    print(f'✅ High Risk list exported to: {hr_path}')

print(f'\n🎉 Dashboard complete!')